In [1]:
from collections import Counter
import argparse
import os
import json

import numpy as np
from pathlib import Path
from tqdm import tqdm
from p_tqdm import p_map
from scipy.stats import wasserstein_distance

from pymatgen.core.structure import Structure
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
from matminer.featurizers.composition.composite import ElementProperty

from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
from pymatgen.io.cif import CifWriter

#added by Tsach


from eval_utils import (
    smact_validity, structure_validity, CompScaler, get_fp_pdist,
    load_config, load_data, get_crystals_list, prop_model_eval, compute_cov)


CrystalNNFP = CrystalNNFingerprint.from_preset("ops")
CompFP = ElementProperty.from_preset('magpie')

Percentiles = {
    'mp20': np.array([-3.17562208, -2.82196882, -2.52814761]),
    'carbon': np.array([-154.527093, -154.45865733, -154.44206825]),
    'perovskite': np.array([0.43924842, 0.61202443, 0.7364607]),
}

COV_Cutoffs = {
    'mp20': {'struc': 0.4, 'comp': 10.},
    'carbon': {'struc': 0.2, 'comp': 4.},
    'perovskite': {'struc': 0.2, 'comp': 4},
}

In [2]:
# parser = argparse.ArgumentParser()
# parser.add_argument('--root_path', default='/home/gridsan/tmackey/hydra/singlerun/2023-11-01/og_CDVAE_neg_mask_mp_20') # changed the default to the 
# parser.add_argument('--label', default='')
# parser.add_argument('--tasks', nargs='+', default=['recon']) #changing the default to recon only 
# parser.add_argument('--compare_diffraction_patterns', default=False) #no diffraction pattern comparison 
# args = parser.parse_args()

In [3]:
import sys

In [4]:
model_path = '/home/gridsan/tmackey/hydra/singlerun/2023-10-29/no_encode_intensity_concat_comp_concat_neg_mask_mp_20'
num_batches = 1

In [5]:
all_metrics = {}

cfg = load_config(model_path)
eval_model_name = cfg.data.eval_model_name

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


In [6]:
def get_file_paths(root_path, task, label='', suffix='pt'):
    if label == '':
        out_name = f'eval_{task}.{suffix}'
    else:
        out_name = f'eval_{task}_{label}.{suffix}'
    out_name = os.path.join(root_path, out_name)
    return out_name

recon_file_path = get_file_paths(model_path, 'recon',"")

# def get_crystal_array_list(file_path, batch_idx=0):
file_path = recon_file_path
data = load_data(file_path)

In [7]:
def get_crystals_list(
        frac_coords, atom_types, lengths, angles, num_atoms):
    """
    args:
        frac_coords: (num_atoms, 3)
        atom_types: (num_atoms)
        lengths: (num_crystals)
        angles: (num_crystals)
        num_atoms: (num_crystals)
    """
    assert frac_coords.size(0) == atom_types.size(0) == num_atoms.sum()
    assert lengths.size(0) == angles.size(0) == num_atoms.size(0)

    start_idx = 0
    crystal_array_list = []
    for batch_idx, num_atom in enumerate(num_atoms.tolist()):
        cur_frac_coords = frac_coords.narrow(0, start_idx, num_atom)
        cur_atom_types = atom_types.narrow(0, start_idx, num_atom)
        cur_lengths = lengths[batch_idx]
        cur_angles = angles[batch_idx]

        crystal_array_list.append({
            'frac_coords': cur_frac_coords.detach().cpu().numpy(),
            'atom_types': cur_atom_types.detach().cpu().numpy(),
            'lengths': cur_lengths.detach().cpu().numpy(),
            'angles': cur_angles.detach().cpu().numpy(),
        })
        start_idx = start_idx + num_atom
    return crystal_array_list


In [8]:
def get_crystal_array_list(file_path, batch_idx=0):
    data = load_data(file_path)
    crys_array_list = get_crystals_list(
        data['frac_coords'][batch_idx],
        data['atom_types'][batch_idx],
        data['lengths'][batch_idx],
        data['angles'][batch_idx],
        data['num_atoms'][batch_idx])

    if 'input_data_batch' in data:
        batch = data['input_data_batch']
        if isinstance(batch, dict):
            true_crystal_array_list = get_crystals_list(
                batch['frac_coords'], batch['atom_types'], batch['lengths'],
                batch['angles'], batch['num_atoms'])
        else:
            true_crystal_array_list = get_crystals_list(
                batch.frac_coords, batch.atom_types, batch.lengths,
                batch.angles, batch.num_atoms)
    else:
        true_crystal_array_list = None

    return crys_array_list, true_crystal_array_list

In [9]:
class Crystal(object):
    def __init__(self, crys_array_dict):
        self.frac_coords = crys_array_dict['frac_coords']
        self.atom_types = crys_array_dict['atom_types']
        self.lengths = crys_array_dict['lengths']
        self.angles = crys_array_dict['angles']
        self.dict = crys_array_dict

        self.get_structure()
        self.get_composition()
        self.get_validity()
        self.get_fingerprints()
    def get_structure(self):
        if min(self.lengths.tolist()) < 0:
            self.constructed = False
            self.invalid_reason = 'non_positive_lattice'
        else:
            try:
                self.structure = Structure(
                    lattice=Lattice.from_parameters(
                        *(self.lengths.tolist() + self.angles.tolist())),
                    species=self.atom_types, coords=self.frac_coords, coords_are_cartesian=False)
                self.constructed = True
            except Exception:
                self.constructed = False
                self.invalid_reason = 'construction_raises_exception'
            if self.structure.volume < 0.1:
                self.constructed = False
                self.invalid_reason = 'unrealistically_small_lattice'
    def get_composition(self):
        elem_counter = Counter(self.atom_types)
        composition = [(elem, elem_counter[elem])
                       for elem in sorted(elem_counter.keys())]
        elems, counts = list(zip(*composition))
        counts = np.array(counts)
        counts = counts / np.gcd.reduce(counts)
        self.elems = elems
        self.comps = tuple(counts.astype('int').tolist())
    def get_validity(self):
        self.comp_valid = smact_validity(self.elems, self.comps)
        if self.constructed:
            self.struct_valid = structure_validity(self.structure)
        else:
            self.struct_valid = False
        self.valid = self.comp_valid and self.struct_valid
    def get_fingerprints(self):
        elem_counter = Counter(self.atom_types)
        comp = Composition(elem_counter)
        self.comp_fp = CompFP.featurize(comp)
        try:
            site_fps = [CrystalNNFP.featurize(
                self.structure, i) for i in range(len(self.structure))]
        except Exception:
            # counts crystal as invalid if fingerprint cannot be constructed.
            print('oops')
            self.valid = False
            self.comp_fp = None
            self.struct_fp = None
            return
        self.struct_fp = np.array(site_fps).mean(axis=0)

In [10]:
class RecEval(object):

    def __init__(self, pred_crys, gt_crys, stol=0.5, angle_tol=10, ltol=0.3): #original values of stol=0.5, angle_tol=10, ltol=0.3
        assert len(pred_crys) == len(gt_crys)
        self.matcher = StructureMatcher(
            stol=stol, angle_tol=angle_tol, ltol=ltol)
        self.preds = pred_crys
        self.gts = gt_crys

    def get_match_rate_and_rms(self):
        def process_one(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                rms_dist = self.matcher.get_rms_dist(
                    pred.structure, gt.structure)
                rms_dist = None if rms_dist is None else rms_dist[0]
                return rms_dist
            except Exception:
                return None
        #define a function that gets the diffraction patterns for pred and gt and returns the RMSD between them
        def process_diff_pattern(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                #get the structures
                pred_structure = pred.structure
                gt_structure = gt.structure
                pred_pattern = xrd_calculator.get_pattern(pred_structure)
                gt_pattern = xrd_calculator.get_pattern(gt_structure)

                pred_adjusted_vector = np.zeros(256)
                minimum = min(256, len(pred_pattern.x))
                pred_adjusted_vector[:minimum] = pred_pattern.x[:minimum]

                gt_adjusted_vector = np.zeros(256)
                minimum = min(256, len(gt_pattern.x))
                gt_adjusted_vector[:minimum] = gt_pattern.x[:minimum]
                
                #calculate the RMSD between the two patterns
                print(pred_adjusted_vector)
                print(gt_adjusted_vector)
                rms_dist = np.sqrt(np.mean((pred_adjusted_vector - gt_adjusted_vector)**2))

                return rms_dist
            except Exception:
                return None    

        validity = [c.valid for c in self.preds]
        
        print(validity)

        rms_dists = []
        evaluate_diff_pattern = False
        if evaluate_diff_pattern:
            diff_dists = []
        for i in tqdm(range(len(self.preds))):
            rms_dists.append(process_one(
                self.preds[i], self.gts[i], validity[i]))
            if evaluate_diff_pattern:
                diff_dists.append(process_diff_pattern(self.preds[i], self.gts[i], validity[i]))
        rms_dists = np.array(rms_dists)
        if evaluate_diff_pattern:
            diff_dists = np.array(diff_dists)
            average_diff_dist = diff_dists[diff_dists != None].mean()
            #print out all the diff dists
        else:
            average_diff_dist = None
        match_rate = sum(rms_dists != None) / len(self.preds)
        mean_rms_dist = rms_dists[rms_dists != None].mean()

        return {'match_rate': match_rate,
                'rms_dist': mean_rms_dist,
                'diff_dist': average_diff_dist,
                'rmsd_values': rms_dists}

    def get_metrics(self):
        return self.get_match_rate_and_rms()

In [11]:
from multiprocessing import Pool, cpu_count

def create_crystal(x):
    return Crystal(x)
num_cores = cpu_count()
print(num_cores)

80


In [12]:
pool = Pool(processes = num_cores)

In [13]:
from tqdm import tqdm

In [14]:
gt_crys = []
counter = 0 
__, true_crystal_array_list = get_crystal_array_list(file_path, batch_idx=0)
true_crystal_array_list = true_crystal_array_list[0:256]

for x in tqdm(true_crystal_array_list): 
    gt_crys.append(Crystal(x))

  0%|          | 0/256 [00:00<?, ?it/s]/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/analysis/local_env.py:4141: UserWarning: No oxidation states specified on sites! For better results, set the site oxidation states in the structure.
  warnings.warn(
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/analysis/local_env.py:3935: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  warnings.warn(
100%|██████████| 256/256 [01:17<00:00,  3.31it/s]


In [18]:
num_batches = 64

In [19]:
total_rmsd = []

for eval_num in range(num_batches): 
    print("eval num is ", eval_num)
    crys_array_list, true_crystal_array_list = get_crystal_array_list(file_path, batch_idx=eval_num)

    crys_array_list = crys_array_list[0:256]

    pred_crys = []
    counter = 0 
    for x in tqdm(crys_array_list): 
        pred_crys.append(Crystal(x))
        
        
    rec_evaluator = RecEval(pred_crys, gt_crys)
    recon_metrics = rec_evaluator.get_metrics()
    total_rmsd.append(recon_metrics['rmsd_values'])

eval num is  0


100%|██████████| 256/256 [01:41<00:00,  2.53it/s]


[False, True, True, True, False, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, False, True, False, False, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, False, True, True, True, True, True, True, False, True, True, False, True, True, False, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, True, True, True, False, True, True, False, False, True, True, True, True, True, False, True, True, True, True, True, True, True, True, False, False, True, True, True, True, False, True, True, False, False, False, True, True, Fa

100%|██████████| 256/256 [00:01<00:00, 153.07it/s]


eval num is  1


...

[False, True, True, True, False, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, False, True, False, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, False, True, True, True, True, True, True, False, False, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, False, True, True, True, True, True, True, True, True, False, False, True, False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, True, True, False, True, True, False, True, True, True, True, False,

100%|██████████| 256/256 [00:01<00:00, 170.86it/s]


eval num is  25


 21%|██        | 54/256 [00:20<01:17,  2.60it/s]


KeyboardInterrupt: 

In [20]:
print(total_rmsd)

[array([None, None, None, None, None, None, None, None, 0.4015881951664282,
       None, None, None, None, None, 0.030600054028060236,
       0.2841513743976808, None, None, 0.00875305685410957, None, None,
       None, 0.006073638901139476, None, None, None, None, None, None,
       None, None, None, None, None, 0.0376532950812622, None,
       0.010772675473286423, None, None, None, None, None,
       0.056752540265906065, None, None, None, None, None, None, None,
       0.005896661641926606, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, 0.2827203785308909, None,
       None, None, None, None, None, 0.09941705035377775,
       0.006850470641644567, 0.012574134260850663, None,
       0.012434215683223592, None, None, None, 0.010746947465549465,
       0.008758139435983234, None, None, None, None, None, None, None,
       0.2848199146992077, None, None, 0.012329512144374647, None, None,
       None, None, None, 0.007786321873695979, None, 0.

In [24]:
None is not None

False

In [27]:
total_rmsd

[array([None, None, None, None, None, None, None, None, 0.4015881951664282,
        None, None, None, None, None, 0.030600054028060236,
        0.2841513743976808, None, None, 0.00875305685410957, None, None,
        None, 0.006073638901139476, None, None, None, None, None, None,
        None, None, None, None, None, 0.0376532950812622, None,
        0.010772675473286423, None, None, None, None, None,
        0.056752540265906065, None, None, None, None, None, None, None,
        0.005896661641926606, None, None, None, None, None, None, None,
        None, None, None, None, None, None, None, 0.2827203785308909, None,
        None, None, None, None, None, 0.09941705035377775,
        0.006850470641644567, 0.012574134260850663, None,
        0.012434215683223592, None, None, None, 0.010746947465549465,
        0.008758139435983234, None, None, None, None, None, None, None,
        0.2848199146992077, None, None, 0.012329512144374647, None, None,
        None, None, None, 0.00778632187369

In [44]:
total_rmsd = total_rmsd[1:]

In [45]:
resultarray = []
for crystal_i in tqdm(range(256)): 
    matched = 0
    for result in total_rmsd: 
        if result[crystal_i] != None:
            if result[crystal_i] > 0: 
                matched = matched + 1
                break 
    resultarray.append(matched)

100%|██████████| 256/256 [00:00<00:00, 387073.48it/s]


In [46]:
print(resultarray)

[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1]


In [47]:
resultarray = np.array(resultarray)

In [48]:
total_match_rate = sum(resultarray > 0) / len(resultarray)

In [49]:
total_match_rate

0.36328125

In [34]:
total_rmsd

[0,
 array([None, None, None, None, None, None, None, None, None, None, None,
        0.02095605748437915, None, 0.006205438426517514,
        0.0331959554945032, 0.2805653114060484, 0.2230376536569067, None,
        0.004414100033100944, None, 0.013352872270634824, None,
        0.007351391575735324, None, None, None, None, None, None, None,
        None, None, None, 0.012360210925250753, 0.03060356965043643, None,
        0.00726299765114613, None, 0.008713689690015852, None,
        0.00849252880384163, None, 0.06045334554895085, None, None, None,
        0.006605737537370175, None, None, None, 0.008890273261813565,
        0.03487846204112012, None, None, None, None, None, None, None,
        None, None, None, None, None, None, None, None, None, None, None,
        None, None, 0.10126974026503098, 0.009434981790799591,
        0.011750011016770267, None, None, None, None, None,
        0.009524240983100684, 0.010415815007687075, None, None, None, None,
        None, None, None, 0.2

In [40]:
total_rmsd[1][total_rmsd[1] == None] = 0

In [41]:
total_match_rate = sum(total_rmsd[1] > 0) / len(total_rmsd[1])

In [42]:
total_match_rate

0.26171875

In [47]:
total_rmsd

[array([0, 0, 0, 0.03742930570146147, 0, 0.014595663224615776, 0, 0, 0,
        0.28314356603645136, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0.31278374952480403, 0, 0, 0, 0, 0.07840087351103547,
        0.15300332647344086, 0, 0.09575775328390615, 0.08795717048757665,
        0, 0, 0.055663178332767814, 0, 0, 0, 0.09108839345829327,
        0.13528545423269758, 0, 0, 0, 0.03031177528740625, 0,
        0.08714796042199872, 0.12301125211485267, 0.08976654266404469,
        0.27537504516308087, 0, 0, 0, 0, 0, 0, 0, 0.26937406185790025, 0,
        0, 0, 0, 0, 0.0397650144259513, 0.07040630451877769,
        0.03401610924652302, 0, 0, 0, 0, 0, 0, 1.9564172996745786, 0, 0, 0,
        0, 0, 0.21356870367129768, 0.07531983678883052,
        0.04378734968735407, 0, 0, 0, 0, 0, 0.04323372684019662, 0, 0, 0,
        0.024306288095318027, 0, 0, 0, 0, 0.1669472090722077, 0,
        0.06914003836252404, 0, 0, 0, 0.10612016117603897, 0, 0,
        0.028611283022849054, 0.03615455366508402, 0, 0, 0, 

In [48]:
individual_match_rates = [sum(total_rmsd[index] > 0) / len(total_rmsd[index]) for index in range(len(total_rmsd))]

TypeError: 'bool' object is not iterable

In [46]:
individual_match_rates

[0.30859375,
 0.203125,
 0.203125,
 0.2109375,
 0.23828125,
 0.234375,
 0.21875,
 0.203125]

In [41]:
print(match_rate)

NameError: name 'match_rate' is not defined